In [4]:
AUDIO_FILE = "..\\Dataset (Trim)\\1\\r_105_trimmed.wav"

In [5]:
from faster_whisper import WhisperModel


def transcribe_audio(
    audio_file: str,
    model_size: str = "small",
    device: str = "cpu",
    compute_type: str = "int8",
):
    model = WhisperModel(
        model_size,
        device=device,
        compute_type=compute_type,
    )

    segments, info = model.transcribe(
        audio_file,
        word_timestamps=True,
        vad_filter=True,
    )

    words = []

    for segment in segments:

        if segment.words is None:
            continue

        for word in segment.words:

            words.append({
                "word": word.word.strip(),
                "start": word.start,
                "end": word.end,
                "probability": word.probability,
            })

    return words

c:\Users\alvco\Desktop\Manchester2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
words = transcribe_audio(AUDIO_FILE)

for word in words:
    print(
        f"{word['start']:.3f} - "
        f"{word['end']:.3f}: "
        f"{word['word']}"
    )

0.430 - 0.950: Few
0.950 - 1.290: people
1.290 - 1.570: live
1.570 - 1.750: to
1.750 - 1.890: be
1.890 - 1.990: a
1.990 - 2.270: hundred


### Clean previous transcriptions

Remove every transcription `.txt` file (matched and mismatched) found in a folder and its subfolders, so the batch transcription can be run from scratch.

In [16]:
import tkinter as tk
from tkinter import filedialog


def select_folder(title="Select the root folder containing subfolders with audios"):
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    folder_path = filedialog.askdirectory(title=title, parent=root)
    root.destroy()
    return folder_path


def clean_transcriptions(root_folder):
    """Delete every transcription .txt file (matched and mismatched) under root_folder."""
    root = Path(root_folder)

    removed = 0
    for txt_path in root.rglob("*.txt"):
        txt_path.unlink()
        removed += 1

    print(f"Removed {removed} transcription file(s) from {root_folder}")


clean_transcriptions(select_folder(title="Select the folder to clean transcriptions from"))

Removed 20 transcription file(s) from C:/Users/alvco/Desktop/Manchester2026/Dataset/Prueba


## Batch transcription across subfolders

Select a root folder containing subfolders (e.g. `1`, `2`, ...) with audios sharing the same file names. Each audio is transcribed once per subfolder, the majority word sequence per audio name is taken as the correct transcription, and mismatching transcriptions are ignored. A `.txt` file with time windows and words is stored next to each matching audio.

In [17]:
from collections import Counter
from pathlib import Path


def group_audio_files_by_name(root_folder, extensions=(".wav",)):
    """Group audios sharing the same file name across the immediate subfolders of root_folder."""
    root = Path(root_folder)

    grouped = {}
    for subfolder in sorted(p for p in root.iterdir() if p.is_dir()):
        for audio_path in sorted(subfolder.rglob("*")):
            if audio_path.suffix.lower() not in extensions:
                continue
            grouped.setdefault(audio_path.name, {})[subfolder.name] = audio_path

    return grouped


In [18]:
def words_signature(words):
    """A sequence of the (lowercased) transcribed words, ignoring timestamps and probability."""
    return tuple(word["word"].lower() for word in words)


def write_transcription_txt(txt_path, words):
    # Tab-separated with a header so it can be parsed with csv.reader/pandas (e.g. sep="\t")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("start\tend\tword\n")
        for word in words:
            f.write(f"{word['start']:.3f}\t{word['end']:.3f}\t{word['word']}\n")


def transcribe_folder(root_folder, model_size="small", device="cpu", compute_type="int8"):
    """Transcribe every audio found in the subfolders of root_folder, grouped by file name.

    For each audio name, the word sequence obtained for the majority of the subfolders
    is taken as the correct transcription; subfolders whose transcription doesn't match
    the majority are ignored. Matching transcriptions are stored next to their audio as
    a .txt file with the time windows and words.
    """
    grouped = group_audio_files_by_name(root_folder)

    summary = {}

    for audio_name, subfolder_paths in grouped.items():
        transcriptions = {}
        skipped = []
        for subfolder_name, audio_path in subfolder_paths.items():
            if audio_path.with_suffix(".txt").exists():
                skipped.append(subfolder_name)
                continue
            print(f"Transcribing {audio_path}...")
            transcriptions[subfolder_name] = transcribe_audio(
                str(audio_path),
                model_size=model_size,
                device=device,
                compute_type=compute_type,
            )

        if skipped:
            print(f"{audio_name}: skipping already transcribed subfolders {skipped}")

        if not transcriptions:
            continue

        signature_counts = Counter(
            words_signature(words) for words in transcriptions.values()
        )
        majority_signature, majority_count = signature_counts.most_common(1)[0]
        majority_words = next(
            words for words in transcriptions.values()
            if words_signature(words) == majority_signature
        )

        matched, ignored = [], []
        for subfolder_name, words in transcriptions.items():
            audio_path = subfolder_paths[subfolder_name]
            if words_signature(words) == majority_signature:
                matched.append(subfolder_name)
                txt_path = audio_path.with_suffix(".txt")
            else:
                ignored.append(subfolder_name)
                txt_path = audio_path.with_name(audio_path.stem + "_mismatch.txt")
            write_transcription_txt(txt_path, words)

        summary[audio_name] = {
            "majority_count": majority_count,
            "total": len(transcriptions),
            "matched": matched,
            "ignored": ignored,
            "majority_words": majority_words,
        }

        print(
            f"{audio_name}: {majority_count}/{len(transcriptions)} subfolders matched "
            f"the majority transcription. Ignored: {ignored}"
        )

    return summary


In [19]:
root_folder = select_folder()
summary = transcribe_folder(root_folder)
print("Transcription summary:")
for audio_name, info in summary.items():
    print(
        f"{audio_name}: {info['majority_count']}/{info['total']} subfolders matched "
        f"the majority transcription. Ignored: {info['ignored']}"
    )


Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\2\r_1.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\4\r_1.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\5\r_1.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\6\r_1.wav...
r_1.wav: 2/4 subfolders matched the majority transcription. Ignored: ['2', '6']
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\2\r_2.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\4\r_2.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\5\r_2.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\6\r_2.wav...
r_2.wav: 2/4 subfolders matched the majority transcription. Ignored: ['5', '6']
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\2\r_3.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026\Dataset\Prueba\4\r_3.wav...
Transcribing C:\Users\alvco\Desktop\Manchester2026

### Enforce the majority transcription

For each audio, overwrite the mismatched subfolders' transcription with the words that won the majority comparison, so every subfolder ends up with the same words for that audio.

In [20]:
CONTRACTION_SUFFIXES = {"s", "re", "ve", "ll", "d", "t", "m"}


def merge_contraction_tokens(words):
    """Merge a bare contraction suffix token (e.g. "'s") into the previous word.

    Whisper sometimes splits a contraction like "she's" into two tokens ("she", "'s")
    and sometimes keeps it as a single token ("she's"); this normalizes both cases so
    word counts/alignment don't depend on how the contraction was tokenized.
    """
    merged = []
    for word in words:
        text = str(word["word"])
        start = float(word["start"])
        end = float(word["end"])
        if merged and text.startswith("'") and text[1:].lower() in CONTRACTION_SUFFIXES:
            merged[-1]["word"] += text
            merged[-1]["end"] = end
        else:
            merged.append({"start": start, "end": end, "word": text})
    return merged


def apply_majority_transcription(root_folder, summary):
    """Overwrite mismatched subfolders' words with the majority's words, keeping their own original timestamps.

    Only applied when the mismatched transcription has the same number of (contraction-normalized)
    words as the majority; otherwise the subfolder is left as a mismatch.
    """
    import csv

    grouped = group_audio_files_by_name(root_folder)

    for audio_name, info in summary.items():
        if not info["ignored"]:
            continue

        majority_words = merge_contraction_tokens(info["majority_words"])
        subfolder_paths = grouped[audio_name]

        for subfolder_name in info["ignored"]:
            audio_path = subfolder_paths[subfolder_name]
            txt_path = audio_path.with_suffix(".txt")
            mismatch_path = audio_path.with_name(audio_path.stem + "_mismatch.txt")

            with open(mismatch_path, "r", encoding="utf-8") as f:
                original_words = merge_contraction_tokens(list(csv.DictReader(f, delimiter="\t")))

            if len(original_words) != len(majority_words):
                print(
                    f"{audio_name}/{subfolder_name}: word count differs from majority "
                    f"({len(original_words)} vs {len(majority_words)}), skipping"
                )
                continue

            # Keep each subfolder's own start/end, but replace the word text with the majority's
            fixed_words = [
                {"start": original["start"], "end": original["end"], "word": majority["word"]}
                for original, majority in zip(original_words, majority_words)
            ]

            write_transcription_txt(txt_path, fixed_words)
            mismatch_path.unlink()

        print(f"{audio_name}: applied majority words (original timestamps kept) to {info['ignored']}")


apply_majority_transcription(root_folder, summary)


r_1.wav/2: word count differs from majority (12 vs 5), skipping
r_1.wav: applied majority words (original timestamps kept) to ['2', '6']
r_2.wav/5: word count differs from majority (5 vs 4), skipping
r_2.wav/6: word count differs from majority (5 vs 4), skipping
r_2.wav: applied majority words (original timestamps kept) to ['5', '6']
r_3.wav: applied majority words (original timestamps kept) to ['2', '6']
r_4.wav: applied majority words (original timestamps kept) to ['4']
r_5.wav/2: word count differs from majority (6 vs 5), skipping
r_5.wav: applied majority words (original timestamps kept) to ['2', '6']
